In [1]:
!pip install torch
import torch
!pip install torch ipywidgets
from datasets import DatasetDict , Dataset
from transformers import AutoTokenizer , AutoModelForSequenceClassification , TrainingArguments ,Trainer
import evaluate
import numpy as np
from transformers import DataCollatorWithPadding

In [2]:
from datasets import load_dataset

In [3]:
dataset = load_dataset("shawhin/phishing-site-classification")

LOAD PRETRAINED BERT MODEL

In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# define model path
model_path = "google-bert/bert-base-uncased"

# load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path)

# labels
id2label = {0: "Safe", 1: "Not Safe"}
label2id = {"Safe": 0, "Not Safe": 1}

# load model (FIX included)
model = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    attn_implementation="eager"
)

/opt/anaconda3/envs/hf_env/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
!pip install transformers
!pip install accelerate

In [6]:
# FREEZE ALL BASE MODEL PARAMETERS
for name, param in model.base_model.named_parameters():
    param.requires_grad = False

# UNFREEZE BASE MODEL POOLING LAYERS

for name, param in model.base_model.named_parameters():
    if "pooler" in name:
        param.requires_grad = True

DATA PREPROCESSING 

In [7]:
# DEFINE PREPROCESSING
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)
#  examples["text"] → gets the text column from your dataset 

#PREPROCESS ALL DATASETS
tokenized_data = dataset.map(preprocess_function, batched=True)
#  👉 What is happening?
#     .map() applies your function to entire dataset
#      batched=True → processes multiple examples at once (faster)

Map:   0%|          | 0/2100 [00:00<?, ? examples/s]

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

In [8]:
data_collator = DataCollatorWithPadding(tokenizer = tokenizer)

DEFINE EVALUATION METRICS

In [9]:
!pip install scikit-learn

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [10]:
# load metrics
accuracy = evaluate.load("accuracy")     # to measure correct predictions
auc_score = evaluate.load("roc_auc")     # to measure class separation


def compute_metrics(eval_pred):

    # get predictions and labels
    predictions, labels = eval_pred      # model output + actual answers

    # convert logits to probabilities (softmax)
    probabilities = np.exp(predictions) / np.exp(predictions).sum(-1, keepdims=True)

    # take probability of class 1 (Not Safe)
    positive_class_probs = probabilities[:, 1]   # needed for AUC

    # compute AUC score
    auc = np.round(
        auc_score.compute(
            prediction_scores=positive_class_probs,
            references=labels
        )["roc_auc"], 3
    )

    # get predicted class (highest score)
    predicted_classes = np.argmax(predictions, axis=1)

    # compute accuracy
    acc = np.round(
        accuracy.compute(
            predictions=predicted_classes,
            references=labels
        )["accuracy"], 3
    )

    # return both metrics
    return {"Accuracy": acc, "AUC": auc}

In [11]:
pip install -U "transformers[torch]" accelerate

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached transformers-5.14.1-py3-none-any.whl.metadata (32 kB)


  Using cached huggingface_hub-1.24.0-py3-none-any.whl.metadata (16 kB)


  Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl.metadata (7.3 kB)


  Using cached torch-2.13.0-cp310-cp310-macosx_14_0_arm64.whl.metadata (39 kB)
Using cached transformers-5.14.1-py3-none-any.whl (11.6 MB)
Using cached huggingface_hub-1.24.0-py3-none-any.whl (771 kB)
Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl (3.0 MB)


...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━ 3/4 [transformers]


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━ 3/4 [transformers]


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━ 3/4 [transformers]


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [transformers]



Note: you may need to restart the kernel to use updated packages.


TRAINING PARAMETERS 

In [12]:
import accelerate
import transformers

print(accelerate.__version__)
print(transformers.__version__)

1.14.0
4.44.2


In [13]:
# hyperparameters
lr = 2e-4          # learning rate
batch_size = 8     # batch size
num_epochs = 10    # epochs


#  TrainingArguments --->  This sets training settings for the model
training_args = TrainingArguments(

    output_dir="bert-phishing-classifier_teacher",  # save path

    learning_rate=lr,  # learning rate

    per_device_train_batch_size=batch_size,  # train batch
    per_device_eval_batch_size=batch_size,   # eval batch

    num_train_epochs=num_epochs,  # epochs

    logging_strategy="epoch",  # log each epoch
    eval_strategy="epoch",     # eval each epoch
    save_strategy="epoch",     # save each epoch

    load_best_model_at_end=True,  # keep best model
)

FINE TUNING MODEL

In [15]:
!pip uninstall torch transformers -y
!pip install torch==2.1.2 transformers==4.44.2

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Found existing installation: torch 2.13.0


Uninstalling torch-2.13.0:


  Successfully uninstalled torch-2.13.0


Found existing installation: transformers 5.14.1


...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━ 3/4 [transformers]


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━ 3/4 [transformers]


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━ 3/4 [transformers]


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━ 3/4 [transformers]


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━ 3/4 [transformers]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [transformers]



In [16]:
# Trainer setup
trainer = Trainer(
    model=model,                          # model
    args=training_args,                   # training settings
    train_dataset=tokenized_data["train"],  # train data
    eval_dataset=tokenized_data["test"],    # test data               
    data_collator=data_collator,         # padding handler
    compute_metrics=compute_metrics,     # metrics (accuracy, roc_auc)
)

# Start training
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Auc
1,0.494600,0.421661,0.784000,0.911000
2,0.390700,0.364969,0.818000,0.931000
3,0.380400,0.315649,0.860000,0.939000
4,0.358400,0.452213,0.802000,0.943000
5,0.348400,0.330836,0.864000,0.946000
6,0.353200,0.302488,0.873000,0.949000
7,0.320100,0.289899,0.862000,0.949000
8,0.328400,0.296845,0.876000,0.949000
9,0.314600,0.288453,0.867000,0.951000
10,0.305000,0.296818,0.869000,0.951000


TrainOutput(global_step=2630, training_loss=0.35938964379604327, metrics={'train_runtime': 239.811, 'train_samples_per_second': 87.569, 'train_steps_per_second': 10.967, 'total_flos': 706603239165360.0, 'train_loss': 0.35938964379604327, 'epoch': 10.0})

In [18]:
# apply model to validation dataset
predictions = trainer.predict(tokenized_data["validation"])

# Extract the logits and labels from the predictions object
logits = predictions.predictions
labels = predictions.label_ids

# Use your compute_metrics function
metrics = compute_metrics((logits, labels))
print(metrics)



{'Accuracy': 0.893, 'AUC': 0.945}
